# Step by step MP2RAGE preprocessing for freesurfer

In [25]:
import sys
import os
opj = os.path.join
import numpy as np
import nibabel as nib
from pathlib import Path
from nilearn import masking, image

sub = 'sub-01'
deriv_dir = '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/derivatives'
mp2rage_dir = opj(deriv_dir, 'MP2RAGE_source', sub)

# Find inv2
mp2rage_path = Path(mp2rage_dir)
inv2_file = list(mp2rage_path.glob('*inv2*'))[0].relative_to(Path.cwd())
print(inv2_file)
bloop
# # Setup variables
# search_path = "/Applications/MATLAB_R2025b.app/bin"
# pattern = "matlab*"  # Matches anything starting with 'matlab'

# # Create a path object
# path_obj = Path(search_path)

# # Find all matching files
# files = list(path_obj.glob(pattern))

# if files:
#     # Get the first match as a string
#     target_file = str(files[0])
#     print(f"Found file: {target_file}")
# else:
#     print("No matches found.")
# cmd = f'presurf_INV2({})'

ValueError: '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/derivatives/MP2RAGE_source/sub-01/sub-01_MP2RAGE_inv2.nii' is not in the subpath of '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/code/preproc' OR one path is relative and the other is absolute.

In [21]:
os.system(cmd)

sh: -c: line 0: syntax error near unexpected token `('
sh: -c: line 0: `export PATH="$PATH:/Applications/MATLAB_R2025b.app/bin";matlab -batch addpath('/Users/marcusdaghlian/projects/dp-clean-link/pilot-clean/code/preproc/mlab_helpers')'


512

In [ ]:
%%bash
export PATH="$PATH:/Applications/MATLAB_R2025b.app/bin";

# 
# matlab -batch "echo"
echo 

In [ ]:

############################################
# Note: Python implemention of matlab code https://github.com/khanlab/mp2rage_genUniDen.git mp2rage_genUniDen.m
# Date: 2019/09/25
# Author: YingLi Lu
############################################

def MP2RAGErobustfunc(INV1, INV2, beta):
    # matalb: MP2RAGErobustfunc=@(INV1,INV2,beta)(conj(INV1).*INV2-beta)./(INV1.^2+INV2.^2+2*beta);
    return (np.conj(INV1)*INV2-beta)/(INV1**2+INV2**2+2*beta)
def rootsquares_pos(a, b, c):
    # matlab:rootsquares_pos=@(a, b, c)(-b+sqrt(b. ^ 2 - 4 * a.*c))./(2*a)
    return (-b+np.sqrt(b**2 - 4*a*c))/(2*a)

def rootsquares_neg(a, b, c):
    # matlab: rootsquares_neg = @(a, b, c)(-b-sqrt(b. ^ 2 - 4 * a.*c))./(2*a)
    return (-b-np.sqrt(b**2 - 4*a*c))/(2*a)

def mp2rage_genUniDen(MP2RAGE_filenameUNI, MP2RAGE_filenameINV1, MP2RAGE_filenameINV2, MP2RAGE_uniden_output_filename, chosenFactor):
    #########
    # load data
    #########
    MP2RAGEimg = nib.load(MP2RAGE_filenameUNI)
    INV1img = nib.load(MP2RAGE_filenameINV1)
    INV2img = nib.load(MP2RAGE_filenameINV2)

    MP2RAGEimg_img = MP2RAGEimg.get_fdata()
    INV1img_img = INV1img.get_fdata()
    INV2img_img = INV2img.get_fdata()

    if MP2RAGEimg_img.min() >= 0 and MP2RAGEimg_img.max() >= 0.51:
       # converts MP2RAGE to -0.5 to 0.5 scale - assumes that it is getting only positive values
        MP2RAGEimg_img = (
            MP2RAGEimg_img - MP2RAGEimg_img.max()/2)/MP2RAGEimg_img.max()
        integerformat = 1
    else:
        integerformat = 0

    #########
    # computes correct INV1 dataset
    #########
    # gives the correct polarity to INV1
    INV1img_img = np.sign(MP2RAGEimg_img)*INV1img_img

    # because the MP2RAGE INV1 and INV2 is a sum of squares data, while the
    # MP2RAGEimg is a phase sensitive coil combination.. some more maths has to
    # be performed to get a better INV1 estimate which here is done by assuming
    # both INV2 is closer to a real phase sensitive combination

    # INV1pos=rootsquares_pos(-MP2RAGEimg.img,INV2img.img,-INV2img.img.^2.*MP2RAGEimg.img);
    INV1pos = rootsquares_pos(-MP2RAGEimg_img,
                              INV2img_img, -INV2img_img**2*MP2RAGEimg_img)
    INV1neg = rootsquares_neg(-MP2RAGEimg_img,
                              INV2img_img, -INV2img_img**2*MP2RAGEimg_img)

    INV1final = INV1img_img

    INV1final[np.absolute(INV1img_img-INV1pos) > np.absolute(INV1img_img-INV1neg)
              ] = INV1neg[np.absolute(INV1img_img-INV1pos) > np.absolute(INV1img_img-INV1neg)]
    INV1final[np.absolute(INV1img_img-INV1pos) <= np.absolute(INV1img_img-INV1neg)
              ] = INV1pos[np.absolute(INV1img_img-INV1pos) <= np.absolute(INV1img_img-INV1neg)]

    # usually the multiplicative factor shouldn't be greater then 10, but that
    # is not the ase when the image is bias field corrected, in which case the
    # noise estimated at the edge of the imagemight not be such a good measure

    multiplyingFactor = chosenFactor
    noiselevel = multiplyingFactor*np.mean(INV2img_img[:, -11:, -11:])

    # % MP2RAGEimgRobustScanner = MP2RAGErobustfunc(INV1img.img, INV2img.img, noiselevel. ^ 2)
    MP2RAGEimgRobustPhaseSensitive = MP2RAGErobustfunc(
        INV1final, INV2img_img, noiselevel**2)

    if integerformat == 0:
        MP2RAGEimg_img = MP2RAGEimgRobustPhaseSensitive
    else:
        MP2RAGEimg_img = np.round(4095*(MP2RAGEimgRobustPhaseSensitive+0.5))

    #########
    # save image
    #########
    MP2RAGEimg_img = nib.casting.float_to_int(MP2RAGEimg_img,'int16');
    new_MP2RAGEimg = nib.Nifti1Image(MP2RAGEimg_img, MP2RAGEimg.affine, MP2RAGEimg.header)
    nib.save(new_MP2RAGEimg, MP2RAGE_uniden_output_filename)


# - looking at INV2 mask
# mask_img = masking.compute_background_mask(
#     image.threshold_img(inv2, args.threshold)
# )

# combined_data[mask_img.get_fdata() == 0] = -0.5
# masked_img = nib.Nifti1Image(combined_data.astype(np.float32), combined_img.affine)


In [6]:
inv1_path = '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/test_mp2rage/sub-01_MP2RAGE_inv1.nii'
inv2_path = '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/test_mp2rage/sub-01_MP2RAGE_inv2.nii'
inv2spmbc_path = '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/test_mp2rage/presurf_INV2/sub-01_MP2RAGE_inv2_biascorrected.nii'
uni_path = '/Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/pilot-clean/test_mp2rage/sub-01_MP2RAGE_uni.nii'

In [45]:
inv2spmbc = nib.load(inv2spmbc_path)
inv2 = nib.load(inv2_path)
uni = nib.load(uni_path)

In [ ]:
mask_img = masking.compute_background_mask(
    image.threshold_img(inv2spmbc, 70)
    )
# now erode 1 - then dilate 1 
import numpy as np
from scipy import ndimage
from nilearn import image

# 1. Extract the data and the affine from the mask
mask_data = mask_img.get_fdata()
affine = mask_img.affine
tmp_data = mask_data.copy()
for i in range(1):
    # 2. Perform Erosion (removes small noise)
    tmp_data = ndimage.binary_erosion(tmp_data, iterations=2)

    # 3. Perform Dilation (restores the size of the main object)
    # This sequence (Erode then Dilate) is technically an "Opening"
    tmp_data = ndimage.binary_dilation(tmp_data, iterations=2)

# 4. Cast back to a Nifti object
processed_mask_img = image.new_img_like(mask_img, tmp_data.astype(np.int8))


SyntaxError: invalid syntax (1315579278.py, line 24)

In [47]:
nib.save(processed_mask_img, 'inv2_m70e2d2x1.nii')